In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC SUMMARY (Updated with Flattening & Bridging Architecture):
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGHEST VALUE FILTER: Cleans the 'Contract_Amount' string (removes commas/$) 
      and casts to a Double to find contracts >= 1,000,000.
   3. EXCEPTION HANDLING: Uses a 'Placeholder Swap' ([COMMA]) to protect known unit 
      names that contain commas (e.g., 'CAD PB - RESL...') from being improperly split.
   4. FLATTEN LOBs: Columns K and L can contain comma-separated lists. Uses LATERAL 
      VIEW explode(split(...)) to expand these strings into individual evaluation rows.
   5. RESTORE LOBs: Swaps '[COMMA]' back to a real comma after the explosion to match 
      the mapping table perfectly.
   6. TPRM STRING MAPPING & NAME BRIDGE: Maps expanded LOBs to AU IDs by checking if 
      the 'TPRM_Units' string exists inside the parsed LOB strings using SAFE LIKE 
      syntax (replacing the old RLIKE \b logic). Then bridges the Mapping Table's 
      String Name back to the Master List's Numeric BusinessID.
   7. AGGREGATING (NUMERATOR & DENOMINATOR): Counts total DISTINCT engagements 
      (Denominator) and total DISTINCT engagements >= 1MM (Numerator). DISTINCT is 
      crucial to avoid double-counting the flattened rows.
   8. FINAL OUTPUT: Calculates the percentage (Numerator/Denominator * 100), rounds 
      to 2 decimal places, appends a '%' sign, and outputs the strict 6 columns.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- Grab valid TPRM Mapping Strings
    -- Note: Assessable_Unit_ID actually contains the string Name, not the numeric ID
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 2. Extract TP data, clean the contract amount string, and cast to double
    SELECT 
        EngagementNumber,
        
        -- 3. EXCEPTION DICTIONARY: Protect commas in known LOBs using '[COMMA]'
        REPLACE(owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_owning_lob,
        REPLACE(lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_lob_using,
        
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flattened_LOBs AS (
    -- 4 & 5. FLATTEN & RESTORE RULE: Split by commas, explode, then restore the protected commas
    SELECT 
        EngagementNumber, 
        Cleaned_Amount,
        -- RESTORE: Put the actual commas back so it matches the Mapping Table!
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Third_Party_Risk_Data
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 6 & 7. Map engagements, bridge to Master ID, and calculate Num/Denom
    SELECT 
        mast.BusinessID,
        -- Total Distinct Engagements mapped to this AU
        COUNT(DISTINCT f.EngagementNumber) AS Denominator,
        -- Distinct Engagements mapped to this AU that are >= 1 Million
        COUNT(DISTINCT CASE WHEN f.Cleaned_Amount >= 1000000 THEN f.EngagementNumber END) AS Numerator
    FROM Flattened_LOBs f
    
    -- FIXED JOIN: Maps TP engagements to AU IDs (Safe LIKE replaces RLIKE)
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
        
    -- Name Bridge Join: Bridges Mapping Table Name to Master List Numeric ID
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_AU_Name)) = UPPER(TRIM(mast.AU_Name))
    GROUP BY mast.BusinessID
)

-- 8. Final Template: Strict 6-column output anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP02' AS QuestionID,               
    
    -- Math: Handles division by zero safely, calculates percentage, and appends '%'
    CASE 
        WHEN COALESCE(e.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(e.Numerator AS DOUBLE) / e.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 - High Value Traceability with Comma Exceptions
   
   PURPOSE: Verifies Contract Amount cleaning and shows how the Exception 
   Dictionary safely protected comma-containing LOBs during the flattening process.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Flattened AS (
    SELECT 
        p.EngagementNumber,
        p.Contract_Amount AS Raw_Contract_Amount,
        TRY_CAST(REPLACE(REPLACE(TRIM(p.Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount,
        
        -- The original untouched strings
        p.owning_lob AS Full_Col_K,
        p.lob_using AS Full_Col_L,
        
        -- Exception Dictionary applied
        REPLACE(p.owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_owning_lob,
        REPLACE(p.lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_lob_using,